# **MP1b: Mini Project 1**

## Section 1: Overview

**What is the dataset?**

This notebook works with nutritional product data drawn from Open Food Facts, a free, open-source database maintained by a French nonprofit of the same name. The database is built entirely through crowdsourcing: volunteers and producers around the world scan product barcodes and submit nutrition labels, ingredient lists, packaging photos, and country-of-sale information. As of mid-2025, it contains well over three million products across dozens of countries and product categories. Because it is community-maintained rather than curated by a commercial entity, it offers unusually broad global coverage — but it also carries the data quality characteristics you'd expect from a crowdsourced effort: uneven completeness, inconsistent formatting, and occasional outright errors. Understanding those characteristics is part of what this analysis is for.

The data is accessed through Open Food Facts' public search API, which requires no authentication or API key for read operations. Queries are sent to world.openfoodfacts.org (with world.openfoodfacts.net as a staging fallback), filtered to the breakfast cereals category, and paginated until roughly 450 records are collected — comfortably within the 400–600 target range that provides enough volume for meaningful comparisons without overwhelming memory or rate limits. Each API response returns a JSON object per product; those objects are flattened into a pandas DataFrame for analysis. If both API hosts are unavailable, the notebook falls back to a bundled CSV snapshot (cereals_breakfast.csv) so every section runs regardless of network conditions.

The analysis is scoped specifically to breakfast cereals because the category is analytically rich: it contains products ranging from unsweetened oat brans to heavily marketed children's cereals, spans dozens of brands and countries, and sits at the intersection of genuine nutrition science and aggressive health marketing. That combination makes it an ideal testbed for the three questions below.

**Three Analytical Questions**

1. Among breakfast cereals in the database, which brands have the highest average sugar content per 100g, and does a product's Nutri-Score (A–E) reliably rank with its actual sugar level?

2. What share of products that include health-oriented keywords ("whole grain," "natural," "fiber") in their product name have a Nutri-Score of C or worse?

3. Which nutritional fields (e.g., fiber, sodium, saturated fat, energy) are most frequently missing across products in this category, and does data completeness vary by country of origin?



**Why These Questions Matter?**
The practical motivation here is product design. Food and health applications — nutrition trackers, grocery assistants, meal planners, diet management tools — are a large and consequential product category where the quality of underlying nutritional data directly constrains what the user interface can honestly show. When a key field is missing, a product team faces a set of unavoidable fallback choices: display a dash, hide the field entirely, estimate from similar products, or surface a default that may quietly mislead. None of those choices is neutral, and most teams make them without empirical grounding in how common or how patterned the gaps actually are.

The same logic applies to health-claim language. If a nutrition app surfaces search results based on keywords in product names (returning "whole grain" cereals in response to a health filter, for instance) and those names are systematically disconnected from Nutri-Score or sugar content, then the feature is amplifying marketing rather than helping users. Knowing that relationship quantitatively is the kind of evidence that should sit behind a design decision, not emerge from it.

Open Food Facts is not a controlled, curated dataset; it is a living, imperfect, global record of what food products exist and what their labels say. That imperfection is not a reason to avoid it. It is, in fact, part of the point. The real-world messiness of this data is precisely the condition that product teams working in nutrition have to design for, and understanding it empirically is more useful than assuming it away.

## Section 2: Data Profile

In [1]:
import time
import warnings

import pandas as pd
import requests

# Keep notebook output clean from pandas future/deprecation notices.
warnings.filterwarnings("ignore", category=FutureWarning)

# Identify this notebook to the Open Food Facts API.
OFF_USER_AGENT = "HCDE530-assignment/1.0 (https://github.com/openfoodfacts/openfoodfacts-python)"

# Category tag used by the OFF search endpoint.
CATEGORY_TAG = "breakfast-cereals"

# Local CSV snapshot used for fallback/reproducible runs.
SNAPSHOT_CSV = "cereals_breakfast.csv"

# Target sample size (assignment plan: ~400-600 products).
TARGET_FETCH = 450

# Primary + backup OFF hosts; try in order if one fails.
OFF_BASES = [
    "https://world.openfoodfacts.org",
    "https://world.openfoodfacts.net",
]

In [2]:
def flatten_product(p: dict):
    """Map one OFF product JSON object to a flat row for pandas."""
    name = p.get("product_name") or p.get("product_name_en")
    if not name:
        return None
    brands = p.get("brands") or ""
    grade = p.get("nutrition_grade_fr")
    countries = p.get("countries") or ""
    n = p.get("nutriments") or {}
    return {
        "product_name": name,
        "brands": brands.split(",")[0].strip() if brands else "",
        "nutrition_grade_fr": grade,
        "countries": countries.split(",")[0].strip() if countries else "",
        "sugars_100g": n.get("sugars_100g"),
        "fiber_100g": n.get("fiber_100g"),
        "salt_100g": n.get("salt_100g"),
        "saturated-fat_100g": n.get("saturated-fat_100g"),
        "energy-kcal_100g": n.get("energy-kcal_100g"),
    }


def _fetch_search_page(session, base: str, page: int, page_size: int) -> dict:
    fields = "product_name,brands,nutrition_grade_fr,nutriments,countries"
    url = (
        f"{base}/cgi/search.pl"
        f"?action=process&json=true&page_size={page_size}&page={page}"
        f"&tagtype_0=categories&tag_contains_0=contains&tag_0={CATEGORY_TAG}"
        f"&fields={fields}"
    )
    r = session.get(url, timeout=90)
    r.raise_for_status()
    text = r.text.strip()
    if not text.startswith("{"):
        raise ValueError(f"non-JSON response from {base}")
    return r.json()


def fetch_cereals(max_products: int, page_size: int = 100, pause_s: float = 0.4):
    """Paginate OFF legacy JSON search for breakfast cereals; try each host in OFF_BASES per page."""
    rows: list[dict] = []
    page = 1
    session = requests.Session()
    session.headers.update({"User-Agent": OFF_USER_AGENT})
    last_base_used = None

    while len(rows) < max_products:
        payload = None
        errors = []
        for base in OFF_BASES:
            try:
                payload = _fetch_search_page(session, base, page, page_size)
                last_base_used = base
                break
            except Exception as err:
                errors.append(f"{base}: {err!s}")

        if payload is None:
            raise RuntimeError("All OFF hosts failed: " + " | ".join(errors))

        products = payload.get("products") or []
        if not products:
            break
        for p in products:
            flat = flatten_product(p)
            if flat:
                rows.append(flat)
            if len(rows) >= max_products:
                break
        page += 1
        time.sleep(pause_s)

    return pd.DataFrame(rows), last_base_used

In [3]:
# Load MP1 dataset: prefer live Open Food Facts pull; use CSV snapshot if every host fails.
try:
    df, host_used = fetch_cereals(max_products=TARGET_FETCH)
    if len(df) == 0:
        raise RuntimeError("API returned no products")
    df.to_csv(SNAPSHOT_CSV, index=False)
    print(
        f"Loaded {len(df)} products via {host_used} and refreshed {SNAPSHOT_CSV} "
        f"(target band ~400–600; fetch cap={TARGET_FETCH})."
    )
except Exception as exc:
    df = pd.read_csv(SNAPSHOT_CSV)
    print(f"API fetch failed ({type(exc).__name__}: {exc}). Using bundled snapshot {SNAPSHOT_CSV} ({len(df)} rows).")

df.shape

Loaded 450 products via https://world.openfoodfacts.net and refreshed cereals_breakfast.csv (target band ~400–600; fetch cap=450).


(450, 9)

In [19]:
# I'm asking: what columns exist, what types does pandas infer, and what do a few cereal rows look like?
# The answer tells me whether nutrition fields parsed as numbers, whether brand/country text looks usable, and rough scale of the dataset before deeper analysis.
df.head()

,product_name,brands,nutrition_grade_fr,countries,sugars_100g,fiber_100g,salt_100g,saturated-fat_100g,energy-kcal_100g,country_norm,nutrition_grade_lower,grade_ordered,grade_label,health_keywords_name
0,cruesly mélange de noix,Quaker,b,Belgium,12.0,10.0,0.00,2.0,462.000000,Belgium,b,2.0,B,False
1,Croustillant Chocolat,Bjorg,c,France,14.0,10.0,0.26,3.4,431.666667,France,c,3.0,C,False
2,Flocons d'avoine,Bjorg,a,Belgium,1.7,11.0,0.02,1.3,363.333333,Belgium,a,1.0,A,False
3,"Muesli Raisin, Figue, Abricot",Bjorg,a,Belgique,14.0,10.0,0.03,0.9,352.000000,Belgium,a,1.0,A,False
4,NESTLE CHOCAPIC Céréales 375g,Nestlé,b,Belgium,19.9,7.7,0.20,1.1,384.000000,Belgium,b,2.0,B,False


In [7]:
# I'm asking: are there missing values already visible per column, and which columns are numeric vs object?
# The answer means I know memory footprint, non-null counts, and whether I need to coerce dtypes before filtering or aggregation.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_name        450 non-null    str    
 1   brands              450 non-null    str    
 2   nutrition_grade_fr  450 non-null    str    
 3   countries           450 non-null    str    
 4   sugars_100g         443 non-null    float64
 5   fiber_100g          442 non-null    float64
 6   salt_100g           440 non-null    float64
 7   saturated-fat_100g  442 non-null    float64
 8   energy-kcal_100g    445 non-null    float64
dtypes: float64(5), str(4)
memory usage: 31.8 KB


In [18]:
# I'm asking: for the numeric nutrition columns, what are typical values and how spread out are sugars, fiber, salt, saturated fat, and energy?
# The answer means I know min/max/mean ranges (e.g. how sweet cereals get) and whether outliers look extreme before I chart brands and Nutri-Score.
df.describe()

,sugars_100g,fiber_100g,salt_100g,saturated-fat_100g,energy-kcal_100g,grade_ordered
count,443.000000,442.000000,440.000000,442.000000,445.000000,444.000000
mean,14.372993,8.355105,0.309316,2.314071,407.667041,2.502252
std,8.330636,3.661420,0.521391,2.083195,52.070984,1.199132
min,0.000000,0.000000,0.000000,0.000000,95.000000,1.000000
25%,8.250000,6.200000,0.022222,0.900000,375.000000,1.000000
50%,15.000000,8.100000,0.130000,1.700000,401.000000,3.000000
75%,20.000000,10.000000,0.500000,3.287500,442.000000,3.000000
max,38.000000,27.500000,8.400000,16.666667,960.000000,5.000000


In [8]:
# I'm asking: which Nutri-Score grades appear most often in this cereal slice?
# The answer tells me how imbalanced the label distribution is (e.g. many C/D vs few A), which affects any later comparison across grades.
df["nutrition_grade_fr"].value_counts(dropna=False)

nutrition_grade_fr
c                 171
a                 138
d                  65
b                  50
e                  20
unknown             5
not-applicable      1
Name: count, dtype: int64

In [9]:
# I'm asking: which nutrition columns are missing most often, i.e. how complete is macronutrient reporting?
# The answer shows where dashboards or apps would lack trustworthy fields (more NaNs ⇒ more placeholders or gaps for users).
nutrition_cols = [
    "sugars_100g",
    "fiber_100g",
    "salt_100g",
    "saturated-fat_100g",
    "energy-kcal_100g",
]
df[nutrition_cols].isnull().sum().sort_values(ascending=False)

salt_100g             10
fiber_100g             8
saturated-fat_100g     8
sugars_100g            7
energy-kcal_100g       5
dtype: int64

### **Interpretation & Synthesis:**
The following is my interpretations after running the operations:

**head:**
The first rows show one cereal per line with text fields (name, brand, grade, country) and numeric per-100g nutrients, and values look plausible enough to analyze (e.g. sugars from ~2 to ~20 g in the sample).

**info:**
The table has 450 rows × 9 columns: four string columns are fully populated, while the five nutrient fields are float64 with a small number of nulls (roughly 5–10 per field).

**describe:**
Sugars average about 14 g/100g (median 15, max 38), energy averages about 408 kcal/100g, and the spread is wide enough to compare brands and Nutri-Scores meaningfully.

**isnull:**
Missingness is highest for salt (10 rows), then fiber/saturated fat (8), sugars (7), energy (5)—low overall (~1–2%), so gaps are unlikely to invalidate charts but still matter for precise UI copy.

This slice is 450 breakfast cereals from Open Food Facts with product metadata plus five per-100g nutrients. Types are clean: text for identity/labels, floats for nutrition. Missing values are sparse and concentrated in nutrient columns, not names or grades, so the dataset is usable for comparison charts; the main quality issue for analysis is inconsistent country strings

### **Analytical Question 1:**
Which brands have highest average sugar_100g, and does Nutri-Score track sugar reliably?

In [20]:
# Block 1 — Q1:
# "Which brands have highest average sugar_100g, and does Nutri-Score track sugar reliably?"

import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("cereals_breakfast.csv")

# Basic cleanup
df["brands"] = df["brands"].astype(str).str.strip()
df["nutrition_grade_fr"] = df["nutrition_grade_fr"].astype(str).str.strip().str.lower()
df["sugars_100g"] = pd.to_numeric(df["sugars_100g"], errors="coerce")

# Keep rows with valid brand + sugar
q1 = df.dropna(subset=["brands", "sugars_100g"]).copy()

# Top brands by average sugar (add count so tiny samples are visible)
brand_sugar = (
    q1.groupby("brands", as_index=False)
      .agg(
          n_products=("product_name", "count"),
          avg_sugars_100g=("sugars_100g", "mean")
      )
      .sort_values("avg_sugars_100g", ascending=False)
)

print("Top 15 brands by average sugar (g/100g):")
print(brand_sugar.head(15).to_string(index=False))

# Nutri-Score reliability vs sugar
grade_order = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
q1_grade = q1[q1["nutrition_grade_fr"].isin(grade_order)].copy()
q1_grade["grade_num"] = q1_grade["nutrition_grade_fr"].map(grade_order)

# Average sugar by grade
grade_sugar = (
    q1_grade.groupby("nutrition_grade_fr", as_index=False)["sugars_100g"]
            .mean()
            .rename(columns={"sugars_100g": "avg_sugars_100g"})
)
grade_sugar["grade_num"] = grade_sugar["nutrition_grade_fr"].map(grade_order)
grade_sugar = grade_sugar.sort_values("grade_num")

print("\nAverage sugar by Nutri-Score grade:")
print(grade_sugar[["nutrition_grade_fr", "avg_sugars_100g"]].to_string(index=False))

# Correlation: higher grade_num (worse score) should ideally mean higher sugar
corr = q1_grade["grade_num"].corr(q1_grade["sugars_100g"], method="spearman")
print(f"\nSpearman correlation (grade_num vs sugars_100g): {corr:.3f}")

# Optional: quick overlap diagnostics using quartiles
q1_grade["sugar_quartile"] = pd.qcut(q1_grade["sugars_100g"], q=4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
cross = pd.crosstab(q1_grade["nutrition_grade_fr"], q1_grade["sugar_quartile"], normalize="index")
print("\nGrade vs sugar quartiles (row-normalized):")
print(cross.round(3).to_string())

Top 15 brands by average sugar (g/100g):
                   brands  n_products  avg_sugars_100g
         Lucien Georgelin           2        33.500000
            Break & Boost           1        30.000000
           Kellogg&#039;s           1        28.000000
                     OREO           1        27.000000
            Nature Valley           1        26.190476
        Crownfield (Lidl)           1        25.100000
   Terres et céréales bio           1        25.000000
                   Gullon           1        25.000000
T&C Terres & Céréales BIO           1        24.000000
                  Leclerc           1        24.000000
           Muesli Crunchy           1        24.000000
                     TREK           1        23.000000
                  Nesquik           1        22.400000
        Maison Pâtissière           1        22.000000
                 KELLOG'S           1        22.000000

Average sugar by Nutri-Score grade:
nutrition_grade_fr  avg_sugars_100g
      

### **Analytical Question 2:**
What share of products with health-oriented keywords in name have Nutri-Score C or worse?

In [21]:
# Block 2 — Q2:
# "What share of products with health-oriented keywords in name have Nutri-Score C or worse?"

import pandas as pd

df = pd.read_csv("cereals_breakfast.csv")

# Cleanup
df["product_name"] = df["product_name"].astype(str).str.strip()
df["nutrition_grade_fr"] = df["nutrition_grade_fr"].astype(str).str.strip().str.lower()

# Keyword filter (case-insensitive)
# Includes "whole grain", "natural", "fiber"/"fibre"
keyword_pattern = r"\b(whole\s*grain|natural|fiber|fibre)\b"
health_named = df[df["product_name"].str.contains(keyword_pattern, case=False, regex=True, na=False)].copy()

# Define C or worse
c_or_worse = {"c", "d", "e"}
known_grades = {"a", "b", "c", "d", "e"}

health_named["is_known_grade"] = health_named["nutrition_grade_fr"].isin(known_grades)
health_known = health_named[health_named["is_known_grade"]].copy()
health_known["is_c_or_worse"] = health_known["nutrition_grade_fr"].isin(c_or_worse)

n_total_keywords = len(health_named)
n_known = len(health_known)
n_c_or_worse = int(health_known["is_c_or_worse"].sum())
share_c_or_worse = (n_c_or_worse / n_known) if n_known else float("nan")

print(f"Products with keywords in name: {n_total_keywords}")
print(f"With known Nutri-Score (A-E): {n_known}")
print(f"C or worse among known grades: {n_c_or_worse}")
print(f"Share C or worse: {share_c_or_worse:.2%}")

# Optional: distribution among keyword-matched products
dist = health_known["nutrition_grade_fr"].value_counts(normalize=True).sort_index()
print("\nNutri-Score distribution among keyword-matched products:")
print((dist * 100).round(2).astype(str) + "%")

Products with keywords in name: 8
With known Nutri-Score (A-E): 8
C or worse among known grades: 6
Share C or worse: 75.00%

Nutri-Score distribution among keyword-matched products:
nutrition_grade_fr
a    25.0%
c    62.5%
d    12.5%
Name: proportion, dtype: str


/var/folders/5g/1v2k1rz567qdgmk1lyf9k73m0000gn/T/ipykernel_2686/1710918603.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  health_named = df[df["product_name"].str.contains(keyword_pattern, case=False, regex=True, na=False)].copy()


### **Analytical Question 3:**
Which nutritional fields are most frequently missing, and does completeness vary by country?

In [22]:
# Block 3 — Q3:
# "Which nutritional fields are most frequently missing, and does completeness vary by country?"

import pandas as pd

df = pd.read_csv("cereals_breakfast.csv")

# Nutritional fields present in this dataset
nutrition_cols = ["sugars_100g", "fiber_100g", "salt_100g", "saturated-fat_100g", "energy-kcal_100g"]

# Ensure numeric parsing (non-numeric -> NaN, counted as missing)
for col in nutrition_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Overall missingness
missing_overall = (
    df[nutrition_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "field"})
)
print("Overall missingness by field (% of products):")
print(missing_overall.to_string(index=False))

# Country-level completeness
df["countries"] = df["countries"].astype(str).str.strip()

# Keep countries with enough products for stable comparisons (adjust threshold if needed)
min_n = 5
country_counts = df["countries"].value_counts()
valid_countries = country_counts[country_counts >= min_n].index

by_country = (
    df[df["countries"].isin(valid_countries)]
    .groupby("countries")[nutrition_cols]
    .apply(lambda g: g.notna().mean() * 100)
    .round(2)
)

# Add average completeness score across nutritional fields
by_country["avg_completeness_pct"] = by_country.mean(axis=1)
by_country = by_country.sort_values("avg_completeness_pct", ascending=False)

print(f"\nCountry-level completeness (%), countries with n >= {min_n}:")
print(by_country.to_string())

# Optional: where missingness is concentrated (field x country)
missing_country_field = (
    df[df["countries"].isin(valid_countries)]
    .groupby("countries")[nutrition_cols]
    .apply(lambda g: g.isna().mean() * 100)
    .round(2)
)
print("\nMissingness (%) by country and field:")
print(missing_country_field.to_string())

Overall missingness by field (% of products):
             field  missing_pct
         salt_100g     2.222222
        fiber_100g     1.777778
saturated-fat_100g     1.777778
       sugars_100g     1.555556
  energy-kcal_100g     1.111111

Country-level completeness (%), countries with n >= 5:
                sugars_100g  fiber_100g  salt_100g  saturated-fat_100g  energy-kcal_100g  avg_completeness_pct
countries                                                                                                     
Belgique             100.00      100.00     100.00              100.00            100.00               100.000
Belgium              100.00      100.00     100.00              100.00            100.00               100.000
Bélgica              100.00      100.00     100.00              100.00            100.00               100.000
Ireland              100.00      100.00     100.00              100.00            100.00               100.000
Portugal             100.00      100.00 

## Section 3: Analysis

In [13]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

CSV_PATH = Path("cereals_breakfast.csv")
df = pd.read_csv(CSV_PATH)

# Numeric nutrients (missing cells become NaN)
NUM_COLS = [
    "sugars_100g",
    "fiber_100g",
    "salt_100g",
    "saturated-fat_100g",
    "energy-kcal_100g",
]
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Country normalization (raw Open Food Facts strings → canonical English) ---
_COUNTRY_RAW_TO_CANONICAL = {
    "France": "France",
    "Belgium": "Belgium",
    "Belgien": "Belgium",
    "Belgique": "Belgium",
    "Bélgica": "Belgium",
    "United Kingdom": "United Kingdom",
    "en:United Kingdom": "United Kingdom",
    "en:united-kingdom": "United Kingdom",
    "Royaume-Uni": "United Kingdom",
    "Vereinigtes Königreich": "United Kingdom",
    "en:gb": "United Kingdom",
    "gb": "United Kingdom",
    "Spain": "Spain",
    "España": "Spain",
    "Ireland": "Ireland",
    "Irlande": "Ireland",
    "Bulgaria": "Bulgaria",
    "Bugarska": "Bulgaria",
    "Bułgaria": "Bulgaria",
    "Bulgária": "Bulgaria",
    "Austria": "Austria",
    "Áustria": "Austria",
    "Netherlands": "Netherlands",
    "Portugal": "Portugal",
    "Italy": "Italy",
    "Italia": "Italy",
    "Itália": "Italy",
    "Jordan": "Jordan",
    "Armenia": "Armenia",
    "Francia": "France",
    "Frankrijk": "France",
    "Frankreich": "France",
    "Cyprus": "Cyprus",
    "en:france": "France",
    "en:fr": "France",
    "fr": "France",
    "Finland": "Finland",
    "Côte d'Ivoire": "Côte d'Ivoire",
    "Canada": "Canada",
    "Morocco": "Morocco",
    "en:ma": "Morocco",
    "Algérie": "Algeria",
    "Algeria": "Algeria",
    "Deutschland": "Germany",
    "Germany": "Germany",
    "Alemania": "Germany",
    "United States": "United States",
    "Hong Kong": "Hong Kong",
    "Croatia": "Croatia",
    "Hrvatska": "Croatia",
    "Kuwait": "Kuwait",
    "Mexico": "Mexico",
    "Panama": "Panama",
}


def normalize_country(raw: object) -> str:
    if pd.isna(raw) or str(raw).strip() == "":
        return "Unknown"
    key = str(raw).strip()
    return _COUNTRY_RAW_TO_CANONICAL.get(key, key)


df["country_norm"] = df["countries"].map(normalize_country)

# Nutri-Score: keep raw; ordered label for plots (exclude unknown from ordered charts)
_grade_to_order = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
df["nutrition_grade_lower"] = df["nutrition_grade_fr"].astype(str).str.lower().str.strip()
df["grade_ordered"] = df["nutrition_grade_lower"].map(_grade_to_order)
df["grade_label"] = df["nutrition_grade_lower"].map(
    lambda g: {"a": "A", "b": "B", "c": "C", "d": "D", "e": "E"}.get(g, np.nan)
)

print(f"Rows: {len(df)}")
print("Country (normalized) value counts (top 15):")
print(df["country_norm"].value_counts().head(15))



Rows: 450
Country (normalized) value counts (top 15):
country_norm
France            163
Unknown           156
Belgium            67
United Kingdom     16
Portugal           10
Spain               9
Ireland             7
Italy               3
Bulgaria            3
Germany             2
Canada              2
Austria             2
Morocco             2
Armenia             1
Jordan              1
Name: count, dtype: int64


### **Analytical Question 1:**
Among breakfast cereals in the database, which brands have the highest average sugar content per 100g, and does a product's Nutri-Score (A–E) reliably rank with its actual sugar level?

I split question into two ideas — brands and grades. 
1. The bar chart (brands on the side, average sugar on the bottom) shows which brand strings have the highest average sugar per 100 g. It only uses rows with a known Nutri-Score and brands with at least three products, and it shows the top brands by that average so very rare brands do not dominate. 

2. The box plot shows sugar for every product, grouped by Nutri-Score letter (A through E). You can see typical sugar and spread for each letter; if worse letters often go with more sugar, the boxes drift upward toward E. Nutri-Score is not based on sugar alone, so the pattern can be messy. The line chart draws the average sugar per letter in one simple curve. The printed Spearman number sums up, in one statistic, whether “worse” letters tend to pair with more sugar across products with a known grade.

In [14]:
# --- Q1: Brand average sugar vs Nutri-Score vs sugar ---
MIN_PRODUCTS_PER_BRAND = 3

df_known_grade = df[df["grade_label"].notna()].copy()
df_brand = (
    df_known_grade.groupby("brands", as_index=False)
    .agg(n=("sugars_100g", "size"), mean_sugar=("sugars_100g", "mean"))
)
df_brand = df_brand[df_brand["n"] >= MIN_PRODUCTS_PER_BRAND].sort_values(
    "mean_sugar", ascending=False
)
top_k = 18
df_top_brands = df_brand.head(top_k)

fig_brands = px.bar(
    df_top_brands,
    x="mean_sugar",
    y="brands",
    orientation="h",
    text="mean_sugar",
    title=(
        f"Mean sugars (g/100g) by brand — known Nutri-Score only, "
        f"brands with ≥{MIN_PRODUCTS_PER_BRAND} products (top {top_k} by sugar)"
    ),
    labels={"mean_sugar": "Mean sugars (g/100g)", "brands": "Brand"},
    color="mean_sugar",
    color_continuous_scale="Reds",
)
fig_brands.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig_brands.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig_brands.show()

fig_box = px.box(
    df_known_grade,
    x="grade_label",
    y="sugars_100g",
    category_orders={"grade_label": ["A", "B", "C", "D", "E"]},
    title="Sugar content (g/100g) by Nutri-Score — excludes unknown grades",
    labels={"grade_label": "Nutri-Score", "sugars_100g": "Sugars (g/100g)"},
    color="grade_label",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_box.update_layout(showlegend=False)
fig_box.show()

# Ordinal correlation (1=A … 5=E): higher score = worse letter in some sense;
# we expect positive correlation if worse letters align with more sugar.
sub = df_known_grade.dropna(subset=["sugars_100g", "grade_ordered"])
rho = sub["grade_ordered"].corr(sub["sugars_100g"], method="spearman")
print(f"Spearman rho(grade ordinal vs sugars_100g): {rho:.3f} (n={len(sub)})")

grade_means = (
    df_known_grade.dropna(subset=["sugars_100g"])
    .groupby("grade_label", as_index=False)["sugars_100g"]
    .mean()
    .sort_values("grade_label", key=lambda s: s.map({"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}))
)
fig_mean_line = px.line(
    grade_means,
    x="grade_label",
    y="sugars_100g",
    markers=True,
    title="Mean sugar by Nutri-Score (summary trend)",
    labels={"grade_label": "Nutri-Score", "sugars_100g": "Mean sugars (g/100g)"},
)
fig_mean_line.update_xaxes(categoryorder="array", categoryarray=["A", "B", "C", "D", "E"])
fig_mean_line.show()



Spearman rho(grade ordinal vs sugars_100g): 0.739 (n=437)


### **Analytical Question 2:**
What share of products that include health-oriented keywords ("whole grain," "natural," "fiber") in their product name have a Nutri-Score of C or worse?

The code first finds rows whose product name matches those keywords. It then keeps only rows with a known A–E score (unknown is dropped). C or worse means C, D, or E only. The donut chart shows two slices: how many of those products are C–E versus A–B, as percent, label, and count on the chart. The printed lines below repeat the counts and percentages so you can quote them easily; with a small number of matching products, the share can swing a lot, which is worth saying in a report.

In [15]:
# --- Q2: Health keywords in product name vs Nutri-Score C or worse ---
_kw_pat = re.compile(
    r"whole[\s-]?grain|wholegrain|\bnatural\b|fiber|fibre",
    re.IGNORECASE,
)
df["health_keywords_name"] = df["product_name"].astype(str).str.contains(
    _kw_pat, regex=True, na=False
)

# Among keyword hits with known A–E grade only
mask_kw = df["health_keywords_name"] & df["grade_label"].notna()
df_kw = df.loc[mask_kw].copy()
n_kw = len(df_kw)
n_c_or_worse = df_kw["nutrition_grade_lower"].isin(["c", "d", "e"]).sum()
n_better = df_kw["nutrition_grade_lower"].isin(["a", "b"]).sum()

pie_df = pd.DataFrame(
    {
        "segment": ["Nutri-Score C, D, or E", "Nutri-Score A or B"],
        "count": [n_c_or_worse, n_better],
    }
)
pie_df = pie_df[pie_df["count"] > 0]

fig_kw = px.pie(
    pie_df,
    names="segment",
    values="count",
    title=(
        "Among products whose name mentions whole grain / natural / fiber: "
        "share with Nutri-Score C–E vs A–B (unknown grades excluded)"
    ),
    hole=0.45,
    color="segment",
    color_discrete_map={
        "Nutri-Score C, D, or E": "#c0392b",
        "Nutri-Score A or B": "#27ae60",
    },
)
fig_kw.update_traces(textposition="inside", textinfo="percent+label+value")
fig_kw.show()

print(
    f"Products with keyword(s) in name and known grade: {n_kw}\n"
    f"  C–E: {n_c_or_worse} ({100 * n_c_or_worse / n_kw:.1f}%)\n"
    f"  A–B: {n_better} ({100 * n_better / n_kw:.1f}%)"
)



Products with keyword(s) in name and known grade: 11
  C–E: 6 (54.5%)
  A–B: 5 (45.5%)


### **Analytical Question 3:**
Which nutritional fields (e.g., fiber, sodium, saturated fat, energy) are most frequently missing across products in this category, and does data completeness vary by country of origin?

The first bar chart looks at the whole dataset and, for each nutrient column (sugar, fiber, salt, saturated fat, energy per 100 g), shows what share of products has a blank value. Longer bars mean that field is missing more often, so you see which numbers are least complete overall. The heatmap uses normalized country names (so different spellings and codes for the same country are grouped together) and colors how often each field is missing inside each country. Only countries with enough products are included so one-off rows do not define a whole country. Darker cells mean more gaps for that country and that field, which supports comparing data completeness by place as well as which fields are weakest overall.

In [16]:
# --- Q3: Missing nutritional fields; completeness by normalized country ---
MISSING_COLS = NUM_COLS
overall_pct = (
    pd.DataFrame(
        {
            "field": MISSING_COLS,
            "pct_missing": [100 * df[c].isna().mean() for c in MISSING_COLS],
        }
    )
    .sort_values("pct_missing", ascending=True)
)

fig_miss = px.bar(
    overall_pct,
    x="pct_missing",
    y="field",
    orientation="h",
    title="Share of products with missing values (per 100g field)",
    labels={"pct_missing": "% missing", "field": "Field"},
    color="pct_missing",
    color_continuous_scale="Blues",
)
fig_miss.update_layout(showlegend=False)
fig_miss.show()

MIN_PER_COUNTRY = 8
counts = df["country_norm"].value_counts()
keep_countries = counts[counts >= MIN_PER_COUNTRY].index.tolist()
df_ct = df[df["country_norm"].isin(keep_countries)].copy()

miss_by = []
for country in sorted(df_ct["country_norm"].unique()):
    sub = df_ct[df_ct["country_norm"] == country]
    for col in MISSING_COLS:
        miss_by.append(
            {
                "country_norm": country,
                "field": col,
                "pct_missing": 100 * sub[col].isna().mean(),
                "n": len(sub),
            }
        )
heat_df = pd.DataFrame(miss_by)

fig_heat = px.imshow(
    heat_df.pivot(index="country_norm", columns="field", values="pct_missing"),
    labels=dict(x="Field", y="Country (normalized)", color="% missing"),
    title=(
        f"Missing data rate (%) by country — countries with ≥{MIN_PER_COUNTRY} products"
    ),
    color_continuous_scale="YlOrRd",
    aspect="auto",
)
fig_heat.update_layout(xaxis={"side": "bottom"})
fig_heat.show()



## Section 4: Conclusions

**Question 1 — Brands, sugar, and Nutri-Score**
Among brands with at least three products in this sample, Kellogg’s, Nestlé, and similar large labels sit at the high end of average sugar (around 20 g per 100 g), while many smaller or “healthier” positioned brands cluster lower. Nutri-Score lines up with sugar in general: average sugar rises from about 6 g/100g for A to about 25 g/100g for E, and the Spearman correlation is about 0.74—so worse letters usually mean sweeter products. The box plots still show overlap (some B and C cereals are as sweet as D or E products), which suggests Nutri-Score is not a sugar-only label and should not be shown to users as if it were.

What this suggests: 
A nutrition or grocery app can use Nutri-Score as a rough filter, but should pair it with actual sugar (or a nutrient panel) when the goal is “low sugar.” Brand-level rankings are useful only when you require enough products per brand so one-off outliers do not dominate.

Further investigation: 
Merge duplicate brand spellings (e.g. Kellogg’s vs KELLOG'S), break out children’s vs adult/muesli products, and compare sugar to fiber and saturated fat to explain cereals that score worse than their sugar alone would suggest.

**Question 2 — Health keywords in product names**
Only 11 cereals in this slice have “whole grain,” “natural,” or “fiber/fibre” in the product name and a known A–E Nutri-Score. Of those, 6 (about 55%) are C, D, or E—so health-sounding names are not a reliable sign of a good grade here; roughly half look good on paper, half do not. Because the group is so small, this share could change a lot with a bigger sample or different keyword rules, so I would not treat 55% as a stable industry fact.

What this suggests: 
Product teams should not rank or badge “healthy” cereals from name keywords alone. If you surface those search results, show Nutri-Score and key numbers (sugar, fiber) next to the name so the UI reflects label data, not marketing language.

Further investigation: 
Test more keywords and languages, search ingredients and labels (not only product_name), and compare keyword products’ actual sugar and fiber to the rest of the category.

**Question 3 — Missing fields and country**
Across all 450 products, nutrient fields are mostly filled in (roughly 1–2% missing per field). Salt is missing slightly most often, then fiber and saturated fat; sugars and energy are a bit more complete. After normalizing country names, France and Belgium dominate the sample and many rows still have unknown country, so country-level patterns are suggestive but not globally representative. The heatmap shows most large country groups are highly complete, with occasional gaps for specific country–field pairs rather than one nutrient missing everywhere.

What this suggests: 
For breakfast cereals in Open Food Facts, the bigger design problem is messy country metadata and uneven sampling by country, not wholesale missing nutrition. Apps should still plan per-field fallbacks (especially salt and fiber) and avoid country-based features unless countries are normalized and have enough products.

Further investigation: 
Use structured country tags instead of the first token of countries, track missingness over time as crowdsourcing improves, and set minimum product counts per country before showing “typical” nutrient values by region.